# Step 3 — Classification with All 13 Classifiers

This notebook trains and evaluates **all thirteen classifiers** from the extended pipeline:

| # | Classifier | Family | Notes |
|---|---|---|---|
| 1 | MLP | Dense baseline | — |
| 2 | CNN | 1-D conv | — |
| 3 | LSTM | Recurrent | — |
| 4 | BiLSTM | Bidirectional recurrent | — |
| 5 | CNN-BiLSTM | Hybrid | — |
| 6 | Transformer | Attention | — |
| 7 | **XGBoost** | Boosted trees | requires `xgboost` |
| 8 | **TabNet** | Attentive tabular | requires `pytorch-tabnet` |
| 9 | **FT-Transformer** | Tabular transformer | requires `rtdl_revisiting_models` |
| 10 | **TabResNet** | Residual MLP | — |
| 11 | **BlockMoE** | Block-wise MoE | uses block_sizes |
| 12 | **WideDeep** | Linear on PSCV + MLP on PLM | uses block_sizes |
| 13 | **BayesianMLP** | MC-Dropout uncertainty | — |

For every classifier, four protocols run: **self-consistency**, **independent test (70 → 30)**, **5-fold CV**, **10-fold CV**.

### Graceful dependency skip
If `xgboost`, `pytorch-tabnet`, or `rtdl_revisiting_models` is missing, the corresponding classifier is automatically skipped with a warning — the other twelve continue. The first cell installs all three by default; if installation fails on Colab, that classifier just won't run.

### Selecting classifiers
Edit `MODELS_TO_RUN` in the config cell to skip any classifier. Default = all 13.

### Resume on disconnect
After every classifier finishes, results checkpoint to `results.pkl`. If Colab disconnects, just re-run — completed classifiers are skipped.

## Cell 1 — Configuration

In [ ]:
WORK_DIR = '/content/drive/MyDrive/pscv_work'

# Which classifiers to run. Remove names to skip.
MODELS_TO_RUN = [
    # Deep classifiers (always available — only need torch)
    'MLP', 'CNN', 'LSTM', 'BiLSTM', 'CNN-BiLSTM', 'Transformer',
    # Tabular / specialty classifiers (need optional deps; skipped if missing)
    'XGBoost', 'TabNet', 'FT-Transformer', 'TabResNet',
    'BlockMoE', 'WideDeep', 'BayesianMLP',
]

# Which protocols to run per classifier
PROTOCOLS_TO_RUN = ['self_consistency', 'independent', 'cv5', 'cv10']

# Training hyperparameters (applied to all PyTorch classifiers)
EPOCHS = 60
BATCH_SIZE = 64
LR = 1e-3
WEIGHT_DECAY = 1e-5
EARLY_STOPPING_PATIENCE = 10
RANDOM_SEED = 42

import os, json, time, pickle, math, traceback
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from dataclasses import dataclass
from abc import ABC, abstractmethod
from typing import Optional, List

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device :', DEVICE)
if DEVICE == 'cpu':
    print('WARNING: CPU training will be slow. Switch to GPU runtime.')
print('Models :', MODELS_TO_RUN)
print('Protos :', PROTOCOLS_TO_RUN)

## Cell 2 — Install optional dependencies

`xgboost`, `pytorch-tabnet`, and `rtdl_revisiting_models` are needed only by their corresponding classifiers. If any installation fails on your Colab, that classifier is silently skipped later — the others still run.

In [ ]:
!pip install -q xgboost 2>&1 | tail -1
!pip install -q pytorch-tabnet 2>&1 | tail -1
!pip install -q rtdl_revisiting_models 2>&1 | tail -1

# Check what's actually importable
HAVE_XGB = HAVE_TABNET = HAVE_FTT = False
try:
    import xgboost
    HAVE_XGB = True
    print(f'  xgboost {xgboost.__version__} available')
except ImportError:
    print('  xgboost NOT available — XGBoost classifier will be skipped')

try:
    import pytorch_tabnet
    HAVE_TABNET = True
    print(f'  pytorch-tabnet available')
except ImportError:
    print('  pytorch-tabnet NOT available — TabNet will be skipped')

try:
    import rtdl_revisiting_models
    HAVE_FTT = True
    print(f'  rtdl_revisiting_models available')
except ImportError:
    print('  rtdl_revisiting_models NOT available — FT-Transformer will be skipped')

# Map classifier name -> required dep flag
REQUIRES = {
    'XGBoost':        HAVE_XGB,
    'TabNet':         HAVE_TABNET,
    'FT-Transformer': HAVE_FTT,
}

# Filter MODELS_TO_RUN
skipped_for_deps = [m for m in MODELS_TO_RUN if m in REQUIRES and not REQUIRES[m]]
MODELS_TO_RUN = [m for m in MODELS_TO_RUN if (m not in REQUIRES) or REQUIRES[m]]
print()
print('Will run    :', MODELS_TO_RUN)
print('Skipped (deps):', skipped_for_deps)

## Cell 3 — Load cached features and CV splits

In [ ]:
feat_path = os.path.join(WORK_DIR, 'features.npz')
assert os.path.exists(feat_path), f'features.npz not found at {feat_path}. Run Step 2 first.'

d = np.load(feat_path, allow_pickle=True)
X_train = d['X_train']
y_train = d['y_train'].astype(np.float32)
X_indep = d['X_indep']
y_indep = d['y_indep'].astype(np.float32)
block_sizes = d['block_sizes'].tolist()
block_names = [str(n) for n in d['block_names']]

print(f'X_train: {X_train.shape}  pos={int(y_train.sum())}')
print(f'X_indep: {X_indep.shape}  pos={int(y_indep.sum())}')
print(f'Blocks : {list(zip(block_names, block_sizes))}')

with open(os.path.join(WORK_DIR, 'folds_5.json'))  as f: folds_5  = json.load(f)
with open(os.path.join(WORK_DIR, 'folds_10.json')) as f: folds_10 = json.load(f)
print(f'Folds: 5-fold ({len(folds_5)}), 10-fold ({len(folds_10)})')

## Cell 4 — Per-block feature standardization (FeatureCombiner)

In [ ]:
from sklearn.preprocessing import StandardScaler

class FeatureCombiner:
    """Per-block StandardScaler. Refit fresh inside every fold/protocol."""
    def __init__(self, block_sizes, block_names):
        self.block_sizes = list(block_sizes)
        self.block_names = list(block_names)
        self.scalers = [StandardScaler() for _ in self.block_sizes]
        self.fitted = False

    @property
    def block_slices(self):
        s, start = [], 0
        for b in self.block_sizes:
            s.append(slice(start, start + b)); start += b
        return s

    def fit(self, X):
        for sl, sc in zip(self.block_slices, self.scalers):
            sc.fit(X[:, sl])
        self.fitted = True
        return self

    def transform(self, X):
        out = np.empty_like(X, dtype=np.float32)
        for sl, sc in zip(self.block_slices, self.scalers):
            out[:, sl] = sc.transform(X[:, sl])
        return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

    def fit_transform(self, X):
        return self.fit(X).transform(X)


def make_combiner():
    return FeatureCombiner(block_sizes, block_names)

print('FeatureCombiner ready.')

## Cell 5 — Training config + class imbalance weight

In [ ]:
@dataclass
class TrainConfig:
    epochs: int = EPOCHS
    batch_size: int = BATCH_SIZE
    lr: float = LR
    weight_decay: float = WEIGHT_DECAY
    early_stopping_patience: int = EARLY_STOPPING_PATIENCE
    device: str = DEVICE
    pos_weight: float = 1.0


_pos_ratio = float(y_train.mean())
POS_WEIGHT = (1 - _pos_ratio) / max(_pos_ratio, 1e-3)
TRAIN_CFG = TrainConfig(pos_weight=POS_WEIGHT)
print(f'pos_weight = {POS_WEIGHT:.3f} (pos ratio = {_pos_ratio:.3f})')

## Cell 6 — Deep classifier architectures (6 from `models.py`)

MLP, CNN, LSTM, BiLSTM, CNN-BiLSTM, Transformer.

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, in_dim, hidden=(512, 256, 128), dropout=0.3):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.BatchNorm1d(h), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)


class CNN1DClassifier(nn.Module):
    def __init__(self, in_dim, channels=(64, 128, 64), kernel=5, dropout=0.3):
        super().__init__()
        layers, c_prev = [], 1
        for c in channels:
            layers += [nn.Conv1d(c_prev, c, kernel_size=kernel, padding=kernel//2),
                       nn.ReLU(), nn.BatchNorm1d(c), nn.MaxPool1d(2), nn.Dropout(dropout)]
            c_prev = c
        self.conv = nn.Sequential(*layers)
        out_len = in_dim
        for _ in channels: out_len //= 2
        self.head = nn.Sequential(nn.Linear(c_prev * out_len, 128), nn.ReLU(),
                                   nn.Dropout(dropout), nn.Linear(128, 1))
    def forward(self, x):
        z = self.conv(x.unsqueeze(1)).flatten(1)
        return self.head(z).squeeze(-1)


class _SequentialLSTM(nn.Module):
    def __init__(self, in_dim, hidden_size=128, num_layers=2,
                 bidirectional=False, dropout=0.3):
        super().__init__()
        self.bidir = bidirectional
        self.embed = nn.Linear(1, 32)
        self.lstm = nn.LSTM(32, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0,
                            bidirectional=bidirectional)
        feat = hidden_size * (2 if bidirectional else 1)
        self.head = nn.Sequential(nn.Linear(feat, 64), nn.ReLU(),
                                   nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        e = self.embed(x.unsqueeze(-1))
        _, (h, _) = self.lstm(e)
        last = torch.cat([h[-2], h[-1]], dim=-1) if self.bidir else h[-1]
        return self.head(last).squeeze(-1)


class LSTMClassifier(_SequentialLSTM):
    def __init__(self, in_dim, **kw): super().__init__(in_dim, bidirectional=False, **kw)


class BiLSTMClassifier(_SequentialLSTM):
    def __init__(self, in_dim, **kw): super().__init__(in_dim, bidirectional=True, **kw)


class CNNBiLSTMClassifier(nn.Module):
    def __init__(self, in_dim, conv_channels=64, hidden_size=128, dropout=0.3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, conv_channels, 7, padding=3), nn.ReLU(),
            nn.BatchNorm1d(conv_channels), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(conv_channels, conv_channels, 5, padding=2), nn.ReLU(),
            nn.BatchNorm1d(conv_channels), nn.MaxPool1d(2), nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(conv_channels, hidden_size, 2,
                            batch_first=True, bidirectional=True, dropout=dropout)
        self.head = nn.Sequential(nn.Linear(2*hidden_size, 128), nn.ReLU(),
                                   nn.Dropout(dropout), nn.Linear(128, 1))
    def forward(self, x):
        z = self.conv(x.unsqueeze(1)).transpose(1, 2)
        _, (h, _) = self.lstm(z)
        last = torch.cat([h[-2], h[-1]], dim=-1)
        return self.head(last).squeeze(-1)


class TransformerClassifier(nn.Module):
    def __init__(self, in_dim, token_size=16, d_model=128,
                 nhead=4, num_layers=3, dropout=0.2):
        super().__init__()
        self.token_size = token_size
        self.pad = (token_size - in_dim % token_size) % token_size
        self.n_tokens = (in_dim + self.pad) // token_size
        self.proj = nn.Linear(token_size, d_model)
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos = nn.Parameter(torch.zeros(1, self.n_tokens + 1, d_model))
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead, 4*d_model,
                                                dropout, batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 1)
        nn.init.trunc_normal_(self.cls, std=0.02)
        nn.init.trunc_normal_(self.pos, std=0.02)
    def forward(self, x):
        if self.pad: x = nn.functional.pad(x, (0, self.pad))
        b = x.size(0); x = x.view(b, self.n_tokens, self.token_size)
        z = self.proj(x)
        cls = self.cls.expand(b, -1, -1)
        z = torch.cat([cls, z], dim=1) + self.pos
        z = self.encoder(z)
        return self.head(self.norm(z[:, 0])).squeeze(-1)


# Registry of nn.Modules used by NNClassifier
MODEL_REGISTRY = {
    'MLP': MLPClassifier, 'CNN': CNN1DClassifier,
    'LSTM': LSTMClassifier, 'BiLSTM': BiLSTMClassifier,
    'CNN-BiLSTM': CNNBiLSTMClassifier, 'Transformer': TransformerClassifier,
}


def build_nn_model(name, in_dim, **kwargs):
    if name not in MODEL_REGISTRY:
        raise ValueError(f'Unknown nn model {name}')
    return MODEL_REGISTRY[name](in_dim=in_dim, **kwargs)


print('Deep classifier registry:', list(MODEL_REGISTRY))

## Cell 7 — Base `Classifier` ABC + `NNClassifier` wrapper

The Classifier interface lets us drive any model family (PyTorch / XGBoost / TabNet) through the same `fit / predict_proba` API. The protocol runners don't need to know which family they're calling.

In [ ]:
from sklearn.metrics import roc_auc_score


class Classifier(ABC):
    name: str = 'base'

    @abstractmethod
    def fit(self, X_tr, y_tr, X_val, y_val): ...

    @abstractmethod
    def predict_proba(self, X): ...

    def torch_module(self) -> Optional[nn.Module]:
        return None

    def feature_importance(self) -> Optional[np.ndarray]:
        return None


class NNClassifier(Classifier):
    """Wraps an nn.Module with an early-stopping training loop on val AUC."""

    def __init__(self, model_name, in_dim, train_cfg, model_kwargs=None):
        self.name = model_name
        self.in_dim = in_dim
        self.cfg = train_cfg
        self.model_kwargs = model_kwargs or {}
        self.model: Optional[nn.Module] = None

    def _factory(self) -> nn.Module:
        return build_nn_model(self.name, self.in_dim, **self.model_kwargs)

    def fit(self, X_tr, y_tr, X_val, y_val):
        model = self._factory().to(self.cfg.device)
        opt = torch.optim.AdamW(model.parameters(), lr=self.cfg.lr,
                                weight_decay=self.cfg.weight_decay)
        loss_fn = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([self.cfg.pos_weight], device=self.cfg.device)
        )
        ds = TensorDataset(torch.from_numpy(X_tr).float(), torch.from_numpy(y_tr).float())
        loader = DataLoader(ds, batch_size=self.cfg.batch_size, shuffle=True)

        best_auc, best_state, stale = -1.0, None, 0
        X_val_t = torch.from_numpy(X_val).float().to(self.cfg.device)
        for ep in range(self.cfg.epochs):
            model.train()
            for xb, yb in loader:
                xb, yb = xb.to(self.cfg.device), yb.to(self.cfg.device)
                opt.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                opt.step()
            model.eval()
            with torch.no_grad():
                pv = torch.sigmoid(model(X_val_t)).cpu().numpy()
            try:
                auc = roc_auc_score(y_val, pv)
            except Exception:
                auc = 0.0
            if auc > best_auc:
                best_auc = auc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                stale = 0
            else:
                stale += 1
                if stale >= self.cfg.early_stopping_patience:
                    break
        if best_state is not None:
            model.load_state_dict(best_state)
        self.model = model
        return self

    @torch.no_grad()
    def predict_proba(self, X):
        assert self.model is not None
        self.model.eval()
        out = []
        for i in range(0, len(X), 256):
            xb = torch.from_numpy(X[i:i+256]).float().to(self.cfg.device)
            out.append(torch.sigmoid(self.model(xb)).cpu().numpy())
        return np.concatenate(out)

    def torch_module(self):
        return self.model


print('Classifier ABC + NNClassifier wrapper ready.')

## Cell 8 — Exotic classifiers (XGBoost, TabNet, FT-Transformer, TabResNet, BlockMoE, WideDeep, BayesianMLP)

In [ ]:
# =========================================================
# 1) XGBoost
# =========================================================
class XGBoostClassifier(Classifier):
    name = 'XGBoost'
    def __init__(self, in_dim, train_cfg):
        self.in_dim = in_dim
        self.cfg = train_cfg
        self.model = None
    def fit(self, X_tr, y_tr, X_val, y_val):
        import xgboost as xgb
        pos = y_tr.sum(); neg = len(y_tr) - pos
        spw = float(neg / max(pos, 1))
        self.model = xgb.XGBClassifier(
            n_estimators=800, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.7, reg_lambda=1.0,
            min_child_weight=2, scale_pos_weight=spw,
            objective='binary:logistic', eval_metric='auc',
            tree_method='hist', early_stopping_rounds=30,
            n_jobs=-1, random_state=0,
        )
        self.model.fit(X_tr, y_tr.astype(int),
                       eval_set=[(X_val, y_val.astype(int))], verbose=False)
        return self
    def predict_proba(self, X):
        return self.model.predict_proba(X)[:, 1]
    def feature_importance(self):
        return None if self.model is None else self.model.feature_importances_


# =========================================================
# 2) TabNet
# =========================================================
class TabNetClassifier_(Classifier):
    name = 'TabNet'
    def __init__(self, in_dim, train_cfg):
        self.in_dim = in_dim
        self.cfg = train_cfg
        self.model = None
    def fit(self, X_tr, y_tr, X_val, y_val):
        from pytorch_tabnet.tab_model import TabNetClassifier as TNC
        self.model = TNC(
            n_d=64, n_a=64, n_steps=5, gamma=1.5,
            n_independent=2, n_shared=2, lambda_sparse=1e-4,
            optimizer_fn=torch.optim.Adam, optimizer_params=dict(lr=2e-2),
            scheduler_params=dict(step_size=20, gamma=0.9),
            scheduler_fn=torch.optim.lr_scheduler.StepLR,
            mask_type='entmax', device_name=self.cfg.device,
            verbose=0, seed=0,
        )
        self.model.fit(
            X_tr, y_tr.astype(int),
            eval_set=[(X_val, y_val.astype(int))], eval_metric=['auc'],
            max_epochs=self.cfg.epochs * 2,
            patience=self.cfg.early_stopping_patience * 2,
            batch_size=min(1024, len(X_tr)), virtual_batch_size=128,
            num_workers=0, drop_last=False,
        )
        return self
    def predict_proba(self, X):
        return self.model.predict_proba(X)[:, 1]
    def feature_importance(self):
        return None if self.model is None else self.model.feature_importances_


# =========================================================
# 3) FT-Transformer (uses rtdl_revisiting_models)
# =========================================================
class _FTWrap(nn.Module):
    def __init__(self, ft):
        super().__init__()
        self.ft = ft
    def forward(self, x):
        return self.ft(x, None).squeeze(-1)


class FTTransformerClassifier(NNClassifier):
    name = 'FT-Transformer'
    def __init__(self, in_dim, train_cfg):
        self.name = 'FT-Transformer'
        self.in_dim = in_dim
        self.cfg = train_cfg
        self.model_kwargs = {}
        self.model = None
    def _factory(self):
        from rtdl_revisiting_models import FTTransformer
        kwargs = FTTransformer.get_default_kwargs()
        return _FTWrap(FTTransformer(
            n_cont_features=self.in_dim,
            cat_cardinalities=[],
            d_out=1,
            **kwargs,
        ))


# =========================================================
# 4) Tabular ResNet
# =========================================================
class _ResNetBlock(nn.Module):
    def __init__(self, d, d_hidden, dropout):
        super().__init__()
        self.norm = nn.BatchNorm1d(d)
        self.lin1 = nn.Linear(d, d_hidden)
        self.act = nn.GELU()
        self.drop1 = nn.Dropout(dropout)
        self.lin2 = nn.Linear(d_hidden, d)
        self.drop2 = nn.Dropout(dropout)
    def forward(self, x):
        z = self.norm(x)
        z = self.act(self.lin1(z))
        z = self.drop1(z)
        z = self.lin2(z)
        z = self.drop2(z)
        return x + z


class TabResNet(nn.Module):
    def __init__(self, in_dim, d=256, d_hidden=512, n_blocks=4, dropout=0.2):
        super().__init__()
        self.proj = nn.Linear(in_dim, d)
        self.blocks = nn.Sequential(*[_ResNetBlock(d, d_hidden, dropout) for _ in range(n_blocks)])
        self.norm = nn.BatchNorm1d(d)
        self.head = nn.Linear(d, 1)
    def forward(self, x):
        z = self.proj(x); z = self.blocks(z); z = self.norm(z)
        return self.head(z).squeeze(-1)


class TabResNetClassifier(NNClassifier):
    name = 'TabResNet'
    def __init__(self, in_dim, train_cfg):
        self.name = 'TabResNet'
        self.in_dim = in_dim
        self.cfg = train_cfg
        self.model_kwargs = {}
        self.model = None
    def _factory(self):
        return TabResNet(self.in_dim)


# =========================================================
# 5) BlockMoE — one expert per feature block, gated
# =========================================================
class _Expert(nn.Module):
    def __init__(self, d_in, d_hidden=256, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden), nn.GELU(),
            nn.BatchNorm1d(d_hidden), nn.Dropout(dropout),
            nn.Linear(d_hidden, d_hidden // 2), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden // 2, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)


class BlockMoE(nn.Module):
    def __init__(self, block_sizes, d_hidden=256, dropout=0.2):
        super().__init__()
        self.block_sizes = block_sizes
        self.slices = []
        s = 0
        for bs in block_sizes:
            self.slices.append(slice(s, s + bs))
            s += bs
        self.experts = nn.ModuleList([
            _Expert(bs, d_hidden=d_hidden, dropout=dropout) for bs in block_sizes
        ])
        gate_in = 2 * len(block_sizes)
        self.gate = nn.Sequential(
            nn.Linear(gate_in, 32), nn.GELU(),
            nn.Linear(32, len(block_sizes)),
        )
    def forward(self, x):
        stats = []
        for sl in self.slices:
            xb = x[:, sl]
            stats.append(xb.mean(dim=1, keepdim=True))
            stats.append(xb.std(dim=1, keepdim=True))
        s = torch.cat(stats, dim=1)
        gate_logits = self.gate(s)
        weights = torch.softmax(gate_logits, dim=1)
        expert_logits = torch.stack(
            [exp(x[:, sl]) for exp, sl in zip(self.experts, self.slices)], dim=1
        )
        return (weights * expert_logits).sum(dim=1)


class BlockMoEClassifier(NNClassifier):
    name = 'BlockMoE'
    def __init__(self, in_dim, train_cfg, block_sizes):
        self.name = 'BlockMoE'
        self.in_dim = in_dim
        self.cfg = train_cfg
        self.block_sizes = block_sizes
        self.model_kwargs = {}
        self.model = None
    def _factory(self):
        return BlockMoE(self.block_sizes)


# =========================================================
# 6) Wide & Deep
# =========================================================
class WideDeep(nn.Module):
    def __init__(self, block_sizes, deep_hidden=(512, 256, 128), dropout=0.3):
        super().__init__()
        assert len(block_sizes) >= 2, 'WideDeep needs ≥2 blocks (PSCV + PLM)'
        self.wide_dim = block_sizes[0]
        self.deep_dim = sum(block_sizes[1:])
        self.wide_slice = slice(0, self.wide_dim)
        self.deep_slice = slice(self.wide_dim, self.wide_dim + self.deep_dim)
        self.wide = nn.Linear(self.wide_dim, 1)
        layers, prev = [], self.deep_dim
        for h in deep_hidden:
            layers += [nn.Linear(prev, h), nn.GELU(),
                       nn.BatchNorm1d(h), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.deep = nn.Sequential(*layers)
    def forward(self, x):
        w = self.wide(x[:, self.wide_slice]).squeeze(-1)
        d = self.deep(x[:, self.deep_slice]).squeeze(-1)
        return w + d


class WideDeepClassifier(NNClassifier):
    name = 'WideDeep'
    def __init__(self, in_dim, train_cfg, block_sizes):
        self.name = 'WideDeep'
        self.in_dim = in_dim
        self.cfg = train_cfg
        self.block_sizes = block_sizes
        self.model_kwargs = {}
        self.model = None
    def _factory(self):
        return WideDeep(self.block_sizes)


# =========================================================
# 7) BayesianMLP — MC-Dropout for predictive uncertainty
# =========================================================
class BayesianMLP(nn.Module):
    def __init__(self, in_dim, hidden=(512, 256, 128), dropout=0.25):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)


class BayesianMLPClassifier(NNClassifier):
    name = 'BayesianMLP'
    def __init__(self, in_dim, train_cfg, T=30):
        self.name = 'BayesianMLP'
        self.in_dim = in_dim
        self.cfg = train_cfg
        self.T = T
        self.model_kwargs = {}
        self.model = None
        self._last_std = None
    def _factory(self):
        return BayesianMLP(self.in_dim)
    @torch.no_grad()
    def predict_proba(self, X):
        assert self.model is not None
        # Keep dropout active, batch-norm fixed
        for m in self.model.modules():
            if isinstance(m, nn.Dropout):
                m.train()
            else:
                m.eval()
        probs_T = []
        for _ in range(self.T):
            out = []
            for i in range(0, len(X), 256):
                xb = torch.from_numpy(X[i:i+256]).float().to(self.cfg.device)
                out.append(torch.sigmoid(self.model(xb)).cpu().numpy())
            probs_T.append(np.concatenate(out))
        probs_T = np.stack(probs_T, axis=0)
        mean = probs_T.mean(axis=0)
        self._last_std = probs_T.std(axis=0)
        return mean
    def predictive_std(self):
        return self._last_std


print('Exotic classifiers defined: XGBoost, TabNet, FT-Transformer, TabResNet, BlockMoE, WideDeep, BayesianMLP')

## Cell 9 — Classifier registry + `build_classifier` factory

Single function builds any of the 13 classifiers. BlockMoE and WideDeep get block_sizes passed automatically.

In [ ]:
# Mapping classifier name -> (requires_blocks, factory)
def _wrap_nn(model_name):
    def f(in_dim, train_cfg, **_):
        return NNClassifier(model_name, in_dim, train_cfg)
    return f


CLASSIFIER_REGISTRY = {
    # Deep nn.Modules
    'MLP':            (False, _wrap_nn('MLP')),
    'CNN':            (False, _wrap_nn('CNN')),
    'LSTM':           (False, _wrap_nn('LSTM')),
    'BiLSTM':         (False, _wrap_nn('BiLSTM')),
    'CNN-BiLSTM':     (False, _wrap_nn('CNN-BiLSTM')),
    'Transformer':    (False, _wrap_nn('Transformer')),
    # New exotic classifiers
    'XGBoost':        (False, lambda in_dim, train_cfg, **_:
                              XGBoostClassifier(in_dim, train_cfg)),
    'TabNet':         (False, lambda in_dim, train_cfg, **_:
                              TabNetClassifier_(in_dim, train_cfg)),
    'FT-Transformer': (False, lambda in_dim, train_cfg, **_:
                              FTTransformerClassifier(in_dim, train_cfg)),
    'TabResNet':      (False, lambda in_dim, train_cfg, **_:
                              TabResNetClassifier(in_dim, train_cfg)),
    'BlockMoE':       (True,  lambda in_dim, train_cfg, block_sizes, **_:
                              BlockMoEClassifier(in_dim, train_cfg, block_sizes)),
    'WideDeep':       (True,  lambda in_dim, train_cfg, block_sizes, **_:
                              WideDeepClassifier(in_dim, train_cfg, block_sizes)),
    'BayesianMLP':    (False, lambda in_dim, train_cfg, **_:
                              BayesianMLPClassifier(in_dim, train_cfg)),
}


def build_classifier(name, in_dim, train_cfg, block_sizes=None, block_names=None):
    if name not in CLASSIFIER_REGISTRY:
        raise ValueError(f'Unknown classifier: {name}')
    needs_blocks, factory = CLASSIFIER_REGISTRY[name]
    if needs_blocks and not block_sizes:
        raise ValueError(f'{name} requires block_sizes')
    return factory(in_dim=in_dim, train_cfg=train_cfg,
                   block_sizes=block_sizes, block_names=block_names)


print('Full classifier registry:', list(CLASSIFIER_REGISTRY))

## Cell 10 — Protocol runners & metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score, matthews_corrcoef, f1_score,
    roc_auc_score, confusion_matrix,
)


def split_internal_val(y, val_frac=0.1, seed=0):
    rng = np.random.RandomState(seed)
    idx = np.arange(len(y))
    rng.shuffle(idx)
    n_val = max(2, int(len(idx) * val_frac))
    pos = idx[y[idx] == 1]
    neg = idx[y[idx] == 0]
    n_val_pos = max(1, int(n_val * y.mean()))
    n_val_neg = max(1, n_val - n_val_pos)
    val_idx = np.concatenate([pos[:n_val_pos], neg[:n_val_neg]])
    tr_idx = np.setdiff1d(idx, val_idx)
    return tr_idx, val_idx


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    y_true_i = y_true.astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_i, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    return {
        'ACC':  accuracy_score(y_true_i, y_pred),
        'MCC':  matthews_corrcoef(y_true_i, y_pred),
        'F1':   f1_score(y_true_i, y_pred, zero_division=0),
        'AUC':  roc_auc_score(y_true_i, y_prob) if len(np.unique(y_true_i)) > 1 else float('nan'),
        'SENS': sens,
        'SPEC': spec,
    }


def make_factory(mname):
    """Return a zero-arg factory that builds a fresh classifier for the current fold."""
    def factory():
        return build_classifier(mname, in_dim=X_train.shape[1],
                                train_cfg=TRAIN_CFG,
                                block_sizes=block_sizes,
                                block_names=block_names)
    return factory


def run_self_consistency(mname, X, y):
    combiner = make_combiner()
    Xs = combiner.fit_transform(X)
    tr_idx, val_idx = split_internal_val(y, seed=RANDOM_SEED)
    clf = make_factory(mname)().fit(Xs[tr_idx], y[tr_idx], Xs[val_idx], y[val_idx])
    return {'y_true': y, 'y_prob': clf.predict_proba(Xs),
            'classifier': clf, 'combiner': combiner}


def run_independent_test(mname, X_tr, y_tr, X_te, y_te):
    combiner = make_combiner()
    Xtr_s = combiner.fit_transform(X_tr)
    Xte_s = combiner.transform(X_te)
    tr_idx, val_idx = split_internal_val(y_tr, seed=RANDOM_SEED)
    clf = make_factory(mname)().fit(Xtr_s[tr_idx], y_tr[tr_idx],
                                     Xtr_s[val_idx], y_tr[val_idx])
    return {'y_true': y_te, 'y_prob': clf.predict_proba(Xte_s),
            'classifier': clf, 'combiner': combiner}


def run_kfold(mname, X, y, folds):
    fold_results = []
    for fold in folds:
        tr_idx = np.array(fold['train_idx'])
        va_idx = np.array(fold['val_idx'])
        Xtr, Xva = X[tr_idx], X[va_idx]
        ytr, yva = y[tr_idx], y[va_idx]
        combiner = make_combiner()
        Xtr_s = combiner.fit_transform(Xtr)
        Xva_s = combiner.transform(Xva)
        ii_tr, ii_val = split_internal_val(ytr, seed=RANDOM_SEED + fold['fold'])
        t0 = time.time()
        clf = make_factory(mname)().fit(Xtr_s[ii_tr], ytr[ii_tr],
                                         Xtr_s[ii_val], ytr[ii_val])
        probs = clf.predict_proba(Xva_s)
        elapsed = time.time() - t0
        try:
            auc = roc_auc_score(yva, probs)
        except Exception:
            auc = float('nan')
        print(f"    fold {fold['fold']}: AUC={auc:.4f}  ({elapsed:.1f}s)")
        fold_results.append({'fold': fold['fold'], 'y_true': yva, 'y_prob': probs})
    return fold_results

print('Protocol runners ready.')

## Cell 11 — Main training loop (all classifiers × all protocols, with checkpointing)

After each classifier completes, results are saved to `results.pkl`. A disconnect or re-run picks up where it left off — completed classifiers are skipped.

If a classifier raises an error (e.g., a transient OOM or a missing optional dep that slipped past Cell 2), it's caught, the error is printed, and the next classifier continues.

In [ ]:
results_path = os.path.join(WORK_DIR, 'results.pkl')

# Resume support
if os.path.exists(results_path):
    with open(results_path, 'rb') as f:
        all_results = pickle.load(f)
    print(f'Resuming. Existing models: {list(all_results)}')
else:
    all_results = {}


for mname in MODELS_TO_RUN:
    if mname in all_results and all(p in all_results[mname] for p in PROTOCOLS_TO_RUN):
        print(f'\n========  {mname}: already complete, skipping  ========')
        continue

    print(f'\n=================  Classifier: {mname}  =================')
    all_results.setdefault(mname, {})

    try:
        if 'self_consistency' in PROTOCOLS_TO_RUN and 'self_consistency' not in all_results[mname]:
            print('\n  [protocol] self-consistency')
            t0 = time.time()
            r = run_self_consistency(mname, X_train, y_train)
            m = compute_metrics(r['y_true'], r['y_prob'])
            print('    metrics:', {k: f'{v:.4f}' for k, v in m.items()})
            print(f'    ({time.time()-t0:.1f}s)')
            all_results[mname]['self_consistency'] = {
                'y_true': r['y_true'], 'y_prob': r['y_prob'], 'metrics': m,
            }

        if 'independent' in PROTOCOLS_TO_RUN and 'independent' not in all_results[mname]:
            print('\n  [protocol] independent test (70 -> 30)')
            t0 = time.time()
            r = run_independent_test(mname, X_train, y_train, X_indep, y_indep)
            m = compute_metrics(r['y_true'], r['y_prob'])
            print('    metrics:', {k: f'{v:.4f}' for k, v in m.items()})
            print(f'    ({time.time()-t0:.1f}s)')
            all_results[mname]['independent'] = {
                'y_true': r['y_true'], 'y_prob': r['y_prob'], 'metrics': m,
            }
            # Save the trained classifier + combiner for Step 4 (boundary/SHAP)
            try:
                with open(os.path.join(WORK_DIR, f'clf_{mname}_independent.pkl'), 'wb') as f:
                    pickle.dump({'classifier': r['classifier'],
                                 'combiner':   r['combiner']}, f)
            except Exception as save_err:
                print(f'    (could not pickle {mname}: {save_err}; Step 4 will retrain)')

        if 'cv5' in PROTOCOLS_TO_RUN and 'cv5' not in all_results[mname]:
            print('\n  [protocol] 5-fold CV')
            t0 = time.time()
            fr = run_kfold(mname, X_train, y_train, folds_5)
            y_pool = np.concatenate([f['y_true'] for f in fr])
            p_pool = np.concatenate([f['y_prob'] for f in fr])
            pooled = compute_metrics(y_pool, p_pool)
            per_fold = [compute_metrics(f['y_true'], f['y_prob']) for f in fr]
            print('    pooled:', {k: f'{v:.4f}' for k, v in pooled.items()})
            print(f'    ({time.time()-t0:.1f}s)')
            all_results[mname]['cv5'] = {
                'folds': fr, 'pooled_metrics': pooled,
                'per_fold_metrics': per_fold,
            }

        if 'cv10' in PROTOCOLS_TO_RUN and 'cv10' not in all_results[mname]:
            print('\n  [protocol] 10-fold CV')
            t0 = time.time()
            fr = run_kfold(mname, X_train, y_train, folds_10)
            y_pool = np.concatenate([f['y_true'] for f in fr])
            p_pool = np.concatenate([f['y_prob'] for f in fr])
            pooled = compute_metrics(y_pool, p_pool)
            per_fold = [compute_metrics(f['y_true'], f['y_prob']) for f in fr]
            print('    pooled:', {k: f'{v:.4f}' for k, v in pooled.items()})
            print(f'    ({time.time()-t0:.1f}s)')
            all_results[mname]['cv10'] = {
                'folds': fr, 'pooled_metrics': pooled,
                'per_fold_metrics': per_fold,
            }

    except Exception as e:
        print(f'\n  [!] {mname} failed: {type(e).__name__}: {e}')
        traceback.print_exc()
        print(f'  (continuing with next classifier — its partial results are kept)')

    # Checkpoint after each classifier
    with open(results_path, 'wb') as f:
        pickle.dump(all_results, f)
    print(f'\n  [checkpoint] -> {results_path}')

print('\n=== All classifiers complete ===')

## Cell 12 — Summary CSV

In [ ]:
def aggregate(per_fold_metrics):
    keys = per_fold_metrics[0].keys()
    out = {}
    for k in keys:
        vals = np.array([m[k] for m in per_fold_metrics], dtype=np.float64)
        out[k] = (float(np.nanmean(vals)), float(np.nanstd(vals)))
    return out


rows = []
for mname, protos in all_results.items():
    for proto, data in protos.items():
        row = {'model': mname, 'protocol': proto}
        if proto in ('cv5', 'cv10'):
            agg = aggregate(data['per_fold_metrics'])
            for k, (m, s) in agg.items():
                row[k] = f'{m:.4f}±{s:.4f}'
            row['n_folds'] = len(data['per_fold_metrics'])
        else:
            for k, v in data['metrics'].items():
                row[k] = f'{v:.4f}'
            row['n_folds'] = 1
        rows.append(row)

summary_df = pd.DataFrame(rows)
proto_order = {'self_consistency': 0, 'independent': 1, 'cv5': 2, 'cv10': 3}
summary_df['_p'] = summary_df['protocol'].map(proto_order)
summary_df = summary_df.sort_values(['model', '_p']).drop(columns='_p').reset_index(drop=True)
summary_csv = os.path.join(WORK_DIR, 'summary.csv')
summary_df.to_csv(summary_csv, index=False)
print(f'Saved -> {summary_csv}\n')
print(summary_df.to_string(index=False))

## Cell 13 — Files written

In [ ]:
print('Files in', WORK_DIR, ':')
for f in sorted(os.listdir(WORK_DIR)):
    full = os.path.join(WORK_DIR, f)
    if os.path.isfile(full):
        size_mb = os.path.getsize(full) / 1e6
        print(f'  {f:50s}  {size_mb:8.2f} MB')

print('\nStep 3 complete. Open Step 4 (analysis & visualization) next.')